# **Aprendizaje por refuerzos** - FrozenLake y LunarLander (Gymnasium)

## Tarea: Implementar Agentes Q-Learning y DQN

### Objetivos:
1. Implementar el algoritmo Q-Learning
2. Implementar el algoritmo DQN
3. Entrenar y evaluar ambos agentes
4. Comparar el rendimiento de ambos enfoques


In [14]:
# Instalar paquetes requeridos
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

%pip install swig matplotlib gymnasium torch pygame


Note: you may need to restart the kernel to use updated packages.


In [15]:
# Importar las bibliotecas
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random


La siguiente celda permite ejecutar un juego de Frozen Lake *determinista* para jugar con el teclado.

Utilize las teclas de dirección (flechas) o asdw para comandar al agente.


In [16]:
def jugar_frozen_lake(env):
    env.reset()
    
    print("Controles:")
    print("W - Arriba")
    print("S - Abajo") 
    print("A - Izquierda")
    print("D - Derecha")
    print("Q - Salir")
    print("Presione cualquier tecla para empezar...")
    
    pygame.init()
    pygame.display.set_caption("FrozenLake - Juego Interactivo")
    
    clock = pygame.time.Clock()
    ejecutando = True
    
    while ejecutando:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                ejecutando = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                    ejecutando = False
                elif event.key == pygame.K_w or event.key == pygame.K_UP:
                    accion = 3  # Arriba
                elif event.key == pygame.K_s or event.key == pygame.K_DOWN:
                    accion = 1  # Abajo
                elif event.key == pygame.K_a or event.key == pygame.K_LEFT:
                    accion = 0  # Izquierda
                elif event.key == pygame.K_d or event.key == pygame.K_RIGHT:
                    accion = 2  # Derecha
                else:
                    continue
                
                observacion, recompensa, terminado, truncado, info = env.step(accion)
                print(f"Acción: {accion}, Recompensa: {recompensa}, Terminado: {terminado}")
                
                if terminado or truncado:
                    print(f"¡Episodio terminado! Recompensa final: {recompensa}")
                    pygame.time.wait(500)
                    env.reset()
        
        clock.tick(60)
    
    pygame.quit()
    env.close()

env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
# Descomente la línea de abajo para jugar interactivamente
#jugar_frozen_lake(env)


La siguiente celda permite jugar al juego no determinista.

In [17]:
env = gym.make('FrozenLake-v1', map_name="4x4", render_mode='human', is_slippery=True)
# Descomente la línea de abajo para jugar interactivamente
#jugar_frozen_lake(env)

La siguiente clase define la interfaz de los agentes que utilizaremos para jugar al Frozen Lake.


In [18]:
from abc import ABC, abstractmethod

class Agente(ABC):
    
    @abstractmethod
    def elegir_accion(self, estado):
        """Elige una acción dada una observación."""
        pass
    
    @abstractmethod
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Aprende de la experiencia."""
        pass

class AgenteAleatorio(Agente):
    """Agente aleatorio que elige acciones al azar."""
    
    def __init__(self, espacio_acciones):
        # Se guarda el espacio de acciones para poder elegir acciones al azar
        self.espacio_acciones = espacio_acciones
    
    def elegir_accion(self, estado):
        return self.espacio_acciones.sample()
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass  # El agente aleatorio no aprende

# Probar el AgenteAleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
estado, _ = env.reset()
accion = agente_aleatorio.elegir_accion(estado)
print(f"✓ AgenteAleatorio creado y probado. Acción: {accion}")
env.close()


✓ AgenteAleatorio creado y probado. Acción: 0


La siguiente celda define una función para evaluar el desempeño de un agente dado.

In [19]:
# Función de Evaluación de Agentes
def evaluar_agente(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    recompensas_totales = []
    victorias = 0
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        
        while True:
            accion = agente.elegir_accion(estado)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            
            if terminado or truncado:
                break
        
        recompensas_totales.append(recompensa_total)
        if recompensa_total > 0:
            victorias += 1
    
    return {
        'recompensas_totales': recompensas_totales,
        'victorias': victorias,
        'tasa_victorias': victorias / num_episodios,
        'recompensa_promedio': np.mean(recompensas_totales),
        'desv_estandar': np.std(recompensas_totales)
    }

def imprimir_resultados_evaluacion(resultados, nombre_agente):
    """Imprime los resultados de evaluación de forma formateada."""
    print(f"\n{nombre_agente} - Resultados de Evaluación:")
    print(f"Tasa de Victorias: {resultados['tasa_victorias']:.1%}")
    print(f"Recompensa Promedio: {resultados['recompensa_promedio']:.3f}")
    print(f"Desviación Estándar: {resultados['desv_estandar']:.3f}")
    print(f"Total de Victorias: {resultados['victorias']}")

# Probar función de evaluación
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()



Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 1.0%
Recompensa Promedio: 0.010
Desviación Estándar: 0.099
Total de Victorias: 1


La siguiente celda define una función para entrenar un agente.

In [20]:
# Función de Entrenamiento de Agentes
def entrenar_agente(agente, env, num_episodios=1000, max_pasos=500, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []
    num_episodios_10 = int(num_episodios / 10)
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)

            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)
        
        if verbose and (episodio + 1) % num_episodios_10 == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
            longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
            print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
    
    return recompensas_episodios, longitudes_episodios

print("✓ Funciones de entrenamiento definidas")



✓ Funciones de entrenamiento definidas


La siguiente celda define el agente de Q-Learning a implementar.

## Q-Learning Algorithm

1. For each state `s` and action `a`, initialize the table entry **Q(s, a)** to zero.  
2. Observe the current state **s**  
3. Repeat forever (or for each episode):
    - Select an action **a** and execute it  
    - Receive immediate reward **r**  
    - Observe the new state **s'**  
    - Update the table entry for **Q(s, a)** as follows:
      $$
      Q(s, a) \leftarrow r + \gamma \max_{a'} Q(s', a')
      $$
    - Set **s ← s'**

## Q-Learning en entornos no deterministas

En entornos estocásticos, ejecutar una acción desde un estado no garantiza llegar siempre al mismo siguiente estado, ni recibir siempre la misma recompensa. Por eso, la actualización de la tabla Q se hace de manera **promediada**, usando un factor de aprendizaje α.

La fórmula clásica de actualización es:

$$
Q(s, a) \leftarrow (1-\alpha) Q(s, a) + \alpha \big[r + \gamma \max_{a'} Q(s', a')\big]
$$

- **α (tasa de aprendizaje):** determina cuánto peso le damos a la nueva información frente a lo aprendido hasta ahora.  
- **r + γ max Q(s', a'):** estimación de la recompensa total esperada al ejecutar la acción.
- En **entornos deterministas**, se puede sobrescribir Q(s, a) directamente porque la acción siempre lleva al mismo estado y recompensa.
- En **entornos no deterministas**, se necesita promediar la experiencia, ya que la acción puede llevar a distintos estados con distintas recompensas. El valor de Q converge así al **valor esperado a largo plazo** de cada acción.


In [21]:
class AgenteQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    
    def __init__(self, espacio_estados, espacio_acciones):
        self.Q_table = np.zeros((espacio_estados.n, espacio_acciones.n))
        self.alpha = 0.8
        self.gamma = 0.9
        self.epsilon = 1
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.espacio_acciones = espacio_acciones
        self.tam_mapa = int(np.sqrt(espacio_estados.n))  

    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy.""" #o softmax, lo cambio a softmax luego 
        if np.random.rand() < self.epsilon:
            return np.random.randint(4)  # Exploración
        else:
            return np.argmax(self.Q_table[estado])  # Explotación, accion que maximiza Q
        
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando la ecuación de Bellman."""
        max_q = np.max(self.Q_table[siguiente_estado])
        self.Q_table[estado, accion] = (1 - self.alpha) * self.Q_table[estado, accion] + self.alpha * (recompensa + self.gamma * max_q)
        if terminado:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    def imprimirQtable(self):
        print("Tabla Q:")
        print(self.Q_table)

    def mostrar_policy(self, env):
        """
        Muestra la política aprendida en el mapa del FrozenLake.
        Flechas solo en casillas 'F' (Frozen); 'H' y 'G' se muestran tal cual.
        """
        # Mapeo de acciones a flechas: 0=izquierda, 1=abajo, 2=derecha, 3=arriba
        acciones_map = ["←", "↓", "→", "↑"]

        # Obtener el mapa original del entorno
        desc = env.unwrapped.desc.astype(str)

        # Crear copia del mapa para mostrar la política
        policy_map = desc.copy()

        # Recorrer filas y columnas del mapa
        for fila in range(self.tam_mapa):
            for col in range(self.tam_mapa):
                estado = fila * self.tam_mapa + col  # convertir fila,col a índice lineal
                if desc[fila, col] == 'F':  # solo actualizar casillas 'Frozen'
                    mejor_accion = np.argmax(self.Q_table[estado])
                    policy_map[fila, col] = acciones_map[mejor_accion]

        print("Policy aprendida:")
        for fila in policy_map:
            print(" ".join(fila))


In [22]:
# Entrenar y Evaluar el Agente Q-Learning en FrozenLake-v1 4x4 
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="ansi",  max_episode_steps=500)

agente = AgenteQLearning(env.observation_space, env.action_space)
estado, _ = env.reset()
print(f"FrozenLake-v1 4x4: {env.render()}")
resultados, longitudes = entrenar_agente(agente, env, num_episodios=100000, verbose=True)
agente.imprimirQtable()
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")
agente.mostrar_policy(env)
env.close()

FrozenLake-v1 4x4: 
SFFF
FHFH
FFFH
HFFG

Episodio 10000: Recompensa Promedio = 0.435, Longitud Promedio = 32.4
Episodio 20000: Recompensa Promedio = 0.458, Longitud Promedio = 33.2
Episodio 30000: Recompensa Promedio = 0.472, Longitud Promedio = 33.5
Episodio 40000: Recompensa Promedio = 0.462, Longitud Promedio = 33.9
Episodio 50000: Recompensa Promedio = 0.454, Longitud Promedio = 33.4
Episodio 60000: Recompensa Promedio = 0.473, Longitud Promedio = 34.4
Episodio 70000: Recompensa Promedio = 0.462, Longitud Promedio = 34.0
Episodio 80000: Recompensa Promedio = 0.449, Longitud Promedio = 33.4
Episodio 90000: Recompensa Promedio = 0.476, Longitud Promedio = 33.9
Episodio 100000: Recompensa Promedio = 0.461, Longitud Promedio = 32.8
Tabla Q:
[[1.97674541e-02 9.39394649e-03 9.69881472e-03 1.12827259e-02]
 [1.45640956e-03 9.17161178e-04 2.15960681e-03 1.15131872e-02]
 [4.18654227e-04 3.58782510e-03 3.97575098e-04 3.29581528e-04]
 [8.48624639e-04 2.78544427e-03 6.87629459e-04 6.02384666e-0

In [23]:
env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
estado, _ = env.reset()
agente = AgenteQLearning(env.observation_space, env.action_space)
print(f"FrozenLake-v1 8x8: {env.render()}")
resultados, longitudes = entrenar_agente(agente, env, num_episodios=10000, verbose=False)
agente.imprimirQtable()
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")

FrozenLake-v1 8x8: 
SFFFFFFF
FFFFFFFF
FFFHFFFF
FFFFFHFF
FFFHFFFF
FHHFFFHF
FHFFHFHF
FFFHFFFG

Tabla Q:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.

Aqui deberán incluir código para entrenar y evaluar el agente de Q-Learning implementado.

La siguiente celda define el agente DQN a implementar. 

In [24]:

class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida):
        super(DQN, self).__init__()
        pass
    
    def forward(self, x):
        pass

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""
    
    def __init__(self):
        pass
    
    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        pass
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass
    


Aqui deberán incluir código para entrenar y evaluar el agente de DQN 
implementado.
